# YOLO11l 알약 검출 학습 (20 epoch)
위에서부터 순서대로 실행하세요. 학습은 학습 셀을 실행할 때 시작됩니다.

코랩 커널에는 로컬 PC 파일이 자동으로 전달되지 않습니다. 먼저 코랩에 저장소를 clone하거나 업로드하고, YOLO 형식 이미지·라벨·data.yaml을 준비하세요. 필요하면 Google Drive를 마운트하세요.

런타임 터미널에서 저장소 루트로 이동해 `uv sync`로 환경을 준비하고 해당 환경을 커널로 선택하세요. 현재 코랩 커널을 유지한다면 `uv pip install --python <커널의 sys.executable 경로> -r pyproject.toml`로 프로젝트 의존성을 설치한 뒤 커널을 재시작하세요. uv 자체도 먼저 설치되어 있어야 합니다.

원본 JSON을 YOLO 라벨로 바꾸는 전처리는 이 노트북에 포함하지 않습니다. 기존 실험에서 만든 데이터셋을 재사용합니다. YAML 내부의 경로도 코랩에서 접근 가능해야 합니다.


In [ ]:
# 1. 저장소 경로 찾기: 자동 탐색에 실패하면 아래 값을 실제 경로로 바꾸세요.
import os
import sys
from pathlib import Path

REPOSITORY_PATH = None  # 예: Path("/content/pill-bbox")
current_dir = Path.cwd().resolve()
candidates = (
    [Path(REPOSITORY_PATH).expanduser().resolve()]
    if REPOSITORY_PATH is not None
    else [
        candidate
        for parent in (current_dir, *current_dir.parents)
        for candidate in (parent, parent / "pill-bbox")
    ]
)
repository_root = next(
    (candidate for candidate in candidates if (candidate / "src" / "config.py").is_file()),
    None,
)
if repository_root is None:
    raise FileNotFoundError("커널에 저장소를 준비한 뒤 REPOSITORY_PATH를 지정하세요.")
if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))
os.environ["WANDB_MODE"] = "disabled"
print(f"저장소: {repository_root}")
print(f"커널 Python: {sys.executable}")

In [ ]:
# 2. 공통 설정과 패키지 불러오기
import torch
import yaml
from ultralytics import YOLO

from src.config import PROJECT_ROOT, get_device

DEVICE = get_device()
print(f"학습 장치: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
if DEVICE == "cpu":
    print("CPU에서는 학습이 느립니다. 코랩 GPU 런타임 연결을 확인하세요.")

In [ ]:
# 3. 실험 설정: 데이터가 Drive에 있다면 DATA_YAML과 OUTPUT_DIR을 바꾸세요.
DATA_YAML = PROJECT_ROOT / "data" / "processed" / "yolo" / "data.yaml"
OUTPUT_DIR = PROJECT_ROOT / "runs" / "detect"
MODEL_WEIGHTS = "yolo11l.pt"
SMOKE_TEST = False  # True이면 학습 데이터 1%로 1 epoch 동작 확인
TRAIN_CONFIG = {
    "epochs": 1 if SMOKE_TEST else 20,
    "fraction": 0.01 if SMOKE_TEST else 1.0,
    "imgsz": 640,
    "batch": 4,  # GPU 메모리가 부족하면 2 또는 1로 줄이세요.
    "seed": 42,
    "deterministic": True,
    "workers": 2,
    "device": DEVICE,
    "project": str(OUTPUT_DIR),
    "name": "yolo11l_smoke" if SMOKE_TEST else "yolo11l_20epochs",
    "exist_ok": False,
    "patience": 0,
    "save": True,
    "plots": True,
    "val": True,
}
print(TRAIN_CONFIG)

In [ ]:
# 4. 데이터 설정 확인: 실제 이미지·라벨 점검은 학습 시작 시에도 수행됩니다.
if not DATA_YAML.is_file():
    raise FileNotFoundError(
        f"YOLO 데이터셋 설정이 없습니다: {DATA_YAML}\n"
        "이미지와 YOLO txt 라벨을 준비하고 DATA_YAML 경로를 수정하세요."
    )
with DATA_YAML.open(encoding="utf-8") as file:
    dataset_config = yaml.safe_load(file)
if not isinstance(dataset_config, dict):
    raise ValueError("data.yaml은 매핑 형식이어야 합니다.")
for key in ("train", "val", "names"):
    if not dataset_config.get(key):
        raise ValueError(f"data.yaml에 비어 있지 않은 {key} 항목이 필요합니다.")
print(f"데이터 YAML: {DATA_YAML}")
print(f"클래스 수: {len(dataset_config['names'])}")
print(f"학습 경로: {dataset_config['train']}")
print(f"검증 경로: {dataset_config['val']}")

## 학습 실행
다음 셀은 사전 학습 가중치를 내려받고 실제 학습을 시작합니다. 기본값은 20 epoch이며, 먼저 `SMOKE_TEST = True`로 설정해 1 epoch를 확인할 수 있습니다. fraction은 학습 데이터에만 적용되므로 검증 데이터는 그대로 사용됩니다.

결과 폴더에는 best.pt, last.pt, results.csv와 그래프가 저장됩니다. 코랩 임시 디스크는 세션 종료 시 사라질 수 있으므로 보관하려면 OUTPUT_DIR을 마운트한 Drive 경로로 지정하세요. 재실행하면 새 결과 폴더가 생성됩니다.


In [ ]:
# 5. 학습: 배치별 예측·손실 계산·역전파·가중치 갱신은 내부 루프에서 수행됩니다.
model = YOLO(MODEL_WEIGHTS)
train_results = model.train(data=str(DATA_YAML.resolve()), **TRAIN_CONFIG)
run_dir = Path(model.trainer.save_dir)
best_weights = run_dir / "weights" / "best.pt"
print(f"학습 결과: {run_dir}")
print(f"최적 가중치: {best_weights}")

In [ ]:
# 6. 최적 가중치로 검증: 별도 테스트셋 평가가 아닌 val 재평가입니다.
if not best_weights.is_file():
    raise FileNotFoundError(f"학습을 먼저 완료하세요: {best_weights}")
best_model = YOLO(str(best_weights))
validation_results = best_model.val(
    data=str(DATA_YAML.resolve()),
    split="val",
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    device=DEVICE,
    workers=TRAIN_CONFIG["workers"],
    project=str(run_dir),
    name="best_validation",
)
print(f"mAP50: {validation_results.box.map50:.4f}")
print(f"mAP50-95: {validation_results.box.map:.4f}")

In [ ]:
# 7. 학습 곡선 보기
from IPython.display import Image, display

results_plot = run_dir / "results.png"
if results_plot.is_file():
    display(Image(filename=str(results_plot)))
print(f"epoch별 기록: {run_dir / 'results.csv'}")